class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        #loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)
        
        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=self.config.num_train_epochs, warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size, per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay, logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )
        
        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  tokenizer=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["test"], 
                  eval_dataset=dataset_samsum_pt["validation"])
        
        trainer.train()
        #save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus_samsum_model"))
        #save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

**This code fine-tunes Google's Pegasus model** (a powerful abstractive summarization model) on the **SAMSum dataset** (a popular dialogue summarization dataset) using Hugging Face's `Trainer` API.

It is a clean, modular implementation commonly seen in end-to-end NLP projects (especially in Indian ML courses/bootcamps). The class takes a configuration object and handles model loading, data preparation, training, and saving.

### 1. Class Structure & `__init__`

```python
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
```

**Purpose**: This follows the **configuration-driven design** pattern (very common in production ML pipelines).

- `ModelTrainerConfig` is typically a `@dataclass` that holds all paths and hyperparameters in one place.
- This makes the code:
  - Easy to modify (change values in config instead of hardcoding)
  - Reusable across experiments
  - Compatible with YAML/argument parsers in larger projects

**Example of what `ModelTrainerConfig` probably looks like**:
```python
@dataclass
class ModelTrainerConfig:
    root_dir: str
    data_path: str
    model_ckpt: str = "google/pegasus-cnn_dailymail"
    num_train_epochs: int = 1
    warmup_steps: int = 500
    per_device_train_batch_size: int = 1
    weight_decay: float = 0.01
    logging_steps: int = 10
    evaluation_strategy: str = "steps"
    eval_steps: int = 500
    gradient_accumulation_steps: int = 16
```

### 2. Step-by-Step Breakdown of `train()` Method

#### Step 2.1: Device Selection

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
```

- Automatically uses **GPU** if available (`cuda`), otherwise falls back to **CPU**.
- **Nuance**: For large models like Pegasus, CPU training will be extremely slow. This code is designed to run on a machine with GPU (your RTX 4060 laptop, cloud GPU, etc.).
- **Best Practice Tip**: You can also use `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")` for more flexibility later.

#### Step 2.2: Load Tokenizer and Model

```python
tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
```

- **`AutoTokenizer`**: Loads the tokenizer that matches the model checkpoint (e.g., `google/pegasus-cnn_dailymail`).
  - Handles special tokens, padding, truncation for Pegasus.
- **`AutoModelForSeq2SeqLM`**: Loads the **Pegasus** model for sequence-to-sequence tasks (encoder-decoder architecture).
  - `.to(device)` moves the entire model to GPU/CPU.
- **Why Pegasus?** It was specifically designed for summarization and performs very well on dialogue summarization tasks like SAMSum.

#### Step 2.3: Data Collator (Very Important)

```python
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
```

- This is **critical** for seq2seq models.
- It dynamically pads batches to the longest sequence in that batch (instead of padding everything to max length).
- It also properly shifts the `labels` (for teacher forcing in decoder).
- Without this, training would be inefficient or incorrect.

#### Step 2.4: Load Preprocessed Dataset

```python
dataset_samsum_pt = load_from_disk(self.config.data_path)
```

- Loads the dataset that was previously saved using `dataset.save_to_disk()`.
- The dataset should be a `DatasetDict` with keys like `"train"`, `"validation"`, `"test"`.
- It expects the data to be **already tokenized** (input_ids, attention_mask, labels).

**Important Observation**:
```python
train_dataset=dataset_samsum_pt["test"], 
eval_dataset=dataset_samsum_pt["validation"]
```

This is **unusual**. Normally you should use:
- `train_dataset=dataset_samsum_pt["train"]`
- `eval_dataset=dataset_samsum_pt["validation"]`

**Why it might be written this way**:
- For quick experimentation/demo (SAMSum test split has only ~819 examples → very fast training).
- Possible copy-paste error from a notebook.

**Recommendation**: Change `"test"` → `"train"` for real training.

#### Step 2.5: TrainingArguments Configuration

```python
trainer_args = TrainingArguments(
    output_dir=self.config.root_dir,
    num_train_epochs=self.config.num_train_epochs,
    warmup_steps=self.config.warmup_steps,
    per_device_train_batch_size=self.config.per_device_train_batch_size,
    per_device_eval_batch_size=self.config.per_device_train_batch_size,
    weight_decay=self.config.weight_decay,
    logging_steps=self.config.logging_steps,
    evaluation_strategy=self.config.evaluation_strategy,
    eval_steps=self.config.eval_steps,
    save_steps=1e6,
    gradient_accumulation_steps=self.config.gradient_accumulation_steps
)
```

**Key Parameters Explained**:

| Parameter                        | Purpose                                                                 | Common Values                  | Notes |
|----------------------------------|-------------------------------------------------------------------------|--------------------------------|-------|
| `output_dir`                     | Where to save logs and checkpoints                                      | —                              | — |
| `num_train_epochs`               | How many full passes over the data                                      | 1–5                            | Pegasus converges fast |
| `warmup_steps`                   | Linear warmup for learning rate                                         | 0–1000                         | Helps stability |
| `per_device_train_batch_size`    | Batch size per GPU/CPU                                                  | 1–4 (Pegasus is heavy)         | Use gradient accumulation to increase effective batch size |
| `weight_decay`                   | L2 regularization                                                       | 0.01                           | Prevents overfitting |
| `logging_steps`                  | How often to log training loss                                          | 10–100                         | — |
| `evaluation_strategy`            | When to run evaluation                                                  | `"steps"` or `"epoch"`         | — |
| `eval_steps`                     | Evaluate every N steps                                                  | 500–2000                       | — |
| `save_steps`                     | Save checkpoint every N steps                                           | `1e6` (very high)              | Effectively disables intermediate saves |
| `gradient_accumulation_steps`    | Accumulate gradients over N steps before optimizer step                 | 8–32                           | Simulates larger batch size |

**Note on `save_steps=1e6`**: This is set very high so the model only saves at the **end** of training (not during). This saves disk space during long runs.

#### Step 2.6: Initialize and Run Trainer

```python
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    tokenizer=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"]
)

trainer.train()
```

- The `Trainer` class handles:
  - Optimizer (AdamW by default)
  - Learning rate scheduler
  - Gradient clipping
  - Logging
  - Mixed precision (if `fp16=True` is added)
  - Evaluation loop
- `trainer.train()` starts the actual training loop.

#### Step 2.7: Save Fine-tuned Model & Tokenizer

```python
model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus_samsum_model"))
tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))
```

- Saves the **fine-tuned weights** + config in a format that can be loaded later with `from_pretrained()`.
- Also saves the tokenizer (very important — you need the exact same tokenizer for inference).

### 3. Complete Flow Summary

1. Load config → 2. Load Pegasus + Tokenizer → 3. Prepare data collator → 4. Load tokenized SAMSum data → 5. Configure training hyperparameters → 6. Train → 7. Save model + tokenizer

### 4. Potential Improvements & Best Practices

| Improvement                        | Why It's Better                                      | How to Implement |
|------------------------------------|------------------------------------------------------|------------------|
| Use `"train"` split                | More data = better model                             | Change `["test"]` to `["train"]` |
| Add `fp16=True`                    | Faster training + less VRAM                          | In `TrainingArguments` |
| Add `gradient_checkpointing=True`  | Saves memory for large models                        | In `TrainingArguments` |
| Add `predict_with_generate=True`   | Proper ROUGE evaluation during training              | In `TrainingArguments` + custom `compute_metrics` |
| Add `save_strategy="epoch"`        | Better checkpoint management                         | In `TrainingArguments` |
| Add `load_best_model_at_end=True`  | Keeps the best performing model                      | + `metric_for_best_model="eval_rouge1"` |
| Add `report_to="wandb"`            | Beautiful experiment tracking                        | Install `wandb` + set in args |

### 5. How to Use This Class

```python
config = ModelTrainerConfig(
    root_dir="artifacts/model_trainer",
    data_path="artifacts/data_transformation/samsum_dataset",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16
)

trainer = ModelTrainer(config)
trainer.train()
```

Would you like me to also explain:
- How the data transformation/preprocessing step before this looks like?
- How to add proper ROUGE evaluation during training?
- How to run inference with the saved model?

Or would you like a cleaned-up/improved version of this code?